# Uncertainty Quantification — Entry Point

Thin driver for the full Uncertainty Quantification + Source Attribution
workflow on one channel's SWAT scenario-output CSV (produced by
`10_SWAT_output_aggregation.ipynb`). All the actual analysis and plotting
logic lives in `uq_toolkit.py`.

To reuse for another channel: change `CSV_PATH` / `CHANNEL_NAME` / `REPLACE_NAME`
below. No other code needs to change -- the toolkit auto-detects the number
of scenarios, periods, SSPs, and climate models present in the file.

In [ ]:
from uq_toolkit import run_full_analysis
import os


## Config — edit these for each new channel

In [ ]:
CSV_PATH = "../SWATPlus Models/SWAT_Outputs/aggregated_results/flow_98/cha98_flo_out_wide52.csv"
CHANNEL_NAME = "cha98"
REPLACE_NAME = "Bagaswoti Station"  # optional: replace channel name in output files
OUTDIR = f"../All_DATA/Uncertainty_Outputs/{CHANNEL_NAME}"
if not os.path.exists(OUTDIR):
    os.makedirs(OUTDIR)


In [ ]:
if __name__ == "__main__":
    results = run_full_analysis(CSV_PATH, CHANNEL_NAME, REPLACE_NAME, OUTDIR)

    print("\n--- Monthly % change summary (source attribution) ---")
    print(results["pct_summary"].round(1).to_string(index=False))

    print("\n--- Variance decomposition (near_future) ---")
    print(results["anova"].loc[results["anova"]["Period"] == "near_future"]
          .round(1).to_string(index=False))


## Batch-processing multiple channels

Runs every channel `10_SWAT_output_aggregation.ipynb`

In [ ]:
CHANNEL_NAMES = {
    "12": "Madi-Dang Diversion",
    "33": "Madi-Dang Diversion",
    "50": "Sikta HW",
    "53": "Marikhola Station",
    "69": "Jalakundi Station",
    "97": "Praganna Badkapath",
    "98": "Bagaswoti Station",
    "100": "Lamatal Re-regulating (Kapilvastu Diversion)",
    "102": "Naumure Dam Axis",
}

AGGREGATED_DIR = "../SWATPlus Models/SWAT_Outputs/aggregated_results"

channel_ids = sorted(
    (f.removeprefix("flow_") for f in os.listdir(AGGREGATED_DIR) if f.startswith("flow_")),
    key=int,
)

all_results = {}
for channel_id in channel_ids:
    channel_name = f"cha{channel_id}"
    csv_path = f"{AGGREGATED_DIR}/flow_{channel_id}/{channel_name}_flo_out_wide52.csv"
    replace_name = CHANNEL_NAMES.get(channel_id, channel_name)

    print(f"--- {channel_name} ({replace_name}) ---")
    all_results[channel_name] = run_full_analysis(
        csv_path, channel_name, replace_name,
        outdir=f"../All_DATA/Uncertainty_Outputs/{channel_name}",
    )

print(f"\nDone: {len(all_results)} channels -> All_DATA/Uncertainty_Outputs/")
